# Independent Static Yaw Misalignment Model
### Constant B0 prior + robust relative-heading states (V7.5)

This notebook is the compact, reproducible entry point for the current **independent** model.

The model deliberately keeps the absolute level simple:

$$
B0_i = C - \theta_{i}^{*}
$$

and permits time variation only when label-free SCADA evidence supports a persistent state change:

$$
\hat y_i(t) = B0_i - \beta\,\Delta r_i(t)$$.

Here, \($\theta_i^{*}$\) is the long-term median of rolling power-vs-vane argmax estimates, while
\($\Delta r_i(t)$\) is a sparse, evidence-weighted relative-heading state correction.

### Current validation result

Reference local run on the three labelled PPP turbines:

| Holdout | Model MAE | Constant MAE | Model RMSE | Constant RMSE |
|---|---:|---:|---:|---:|
| PPP_WTG12 | 0.357 | 0.357 | 0.608 | 0.608 |
| PPP_WTG13 | 0.608 | 1.004 | 0.979 | 1.387 |
| PPP_WTG14 | 0.206 | 0.206 | 0.262 | 0.262 |
| **Macro** | **0.390** | **0.522** | **0.616** | **0.752** |

The important behaviour is structural: WTG12 and WTG14 collapse to the constant prior, while WTG13 receives two persistent state corrections.

This notebook does **not** write a submission CSV.

## 1. Reproducibility and data split

The validation split is strict leave-one-turbine-out (LOTO):

- labelled development turbines: `PPP_WTG12`, `PPP_WTG13`, `PPP_WTG14`;
- unlabeled targets inspected after model fitting: `PPP_WTG17`, `SSS_WTG06`;
- all published turbines may contribute **unlabeled** same-site context to the state detector.

The target/holdout labels are never used to construct their SCADA states.

In [1]:
from pathlib import Path
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

# Find repository root without a machine-specific absolute path.
HERE = Path.cwd().resolve()
ROOT = HERE

if not (ROOT / "src").exists():
    matches = [
        parent
        for parent in [HERE, *HERE.parents]
        if (parent / "src").exists()
    ]
    if not matches:
        raise FileNotFoundError(
            "Could not locate repository root containing src/."
        )
    ROOT = matches[0]

zip_candidates = [
    ROOT / "turbines_data.zip",
    ROOT.parent / "turbines_data.zip",
]
ZIP_PATH = next(
    (path for path in zip_candidates if path.exists()),
    None,
)

if ZIP_PATH is None:
    raise FileNotFoundError(
        "Place turbines_data.zip in the repository root or its parent directory."
    )

sys.path.insert(0, str(ROOT / "src"))

from baseline_loto_ridge import read_turbine, list_turbines, wrap_180
from fleet_context import (
    FleetConfig,
    load_layout,
    bin_turbine,
    circular_median_deg,
)
from yaw_relative_state import RelativeStateConfig
from yaw_model import (
    ModelConfig,
    estimate_theta_star_map,
    build_turbine_bundle,
    apply_site_common_mode_veto,
    fit_global_beta,
    predict_from_bundle,
    score_prediction,
)

TRAIN = ["PPP_WTG12", "PPP_WTG13", "PPP_WTG14"]
TARGETS = ["PPP_WTG17", "SSS_WTG06"]

FLEET = FleetConfig()
RELATIVE = RelativeStateConfig()
MODEL = ModelConfig()

print("Repository:", ROOT)
print("SCADA archive:", ZIP_PATH)

Repository: E:\EnergyHacks\github_release
SCADA archive: E:\EnergyHacks\turbines_data.zip


## 2. Absolute prior: long-term aerodynamic reference

The constant prior is retained as the default prediction because it has very low variance.

For each turbine, the frozen B0 estimator computes a rolling apparent power-optimal vane angle
\($\hat\theta_{\text{argmax}}(t)$\). The long-term turbine-specific reference is

$$
\theta_i^{*} = \operatorname{median}_t \hat\theta_{\text{argmax},i}(t).
$$

The labelled training turbines calibrate one shared absolute offset:

$$
C = \operatorname{mean}_{i\in\text{train}} \left(\bar y_i + \theta_i^{*}\right).
$$

The deployed default is therefore

$$
B0_i=C-\theta_i^{*}.
$$

Dynamic state logic is not allowed to replace this absolute anchor; it can only make sparse corrections around it.

In [2]:
t0 = time.perf_counter()

turbines = list_turbines(str(ZIP_PATH))
raw = {
    turbine: read_turbine(str(ZIP_PATH), turbine)
    for turbine in turbines
}

layout = load_layout(ROOT, turbines)
binned = {
    turbine: bin_turbine(raw[turbine], FLEET)
    for turbine in turbines
}

labels = {
    turbine: (
        raw[turbine]
        .assign(date=pd.to_datetime(raw[turbine]["date"]))
        .groupby("date")["yaw_misalignment_deg"]
        .median()
        .sort_index()
    )
    for turbine in TRAIN
}

theta_star = estimate_theta_star_map(
    raw,
    TRAIN + TARGETS,
)

LOAD_SECONDS = time.perf_counter() - t0

theta_table = (
    pd.Series(theta_star, name="theta_star_deg")
    .rename_axis("turbine")
    .to_frame()
)

display(theta_table.round(3))
print(f"Load + B0 estimation: {LOAD_SECONDS:.1f}s")

,theta_star_deg
turbine,
PPP_WTG12,-3.444
PPP_WTG13,0.352
PPP_WTG14,-4.354
PPP_WTG17,-2.483
SSS_WTG06,-1.513


Load + B0 estimation: 162.2s


## 3. Label-free relative-heading state detector

The dynamic channel uses only SCADA and same-site turbine context.

For target \(i\) and neighbour \(j\):

1. form circular heading differences;
2. remove pair/sector baselines;
3. estimate pair reliability from coverage, residual MAD and distance;
4. robustly aggregate usable pairs into a daily target-relative heading residual;
5. construct a neighbour-only background residual to identify common-mode motion.

Two complementary change detectors are used on the cleaned relative signal:

- a persistent rolling before/after detector;
- a conservative L2 state segmentation used only for long two-sided regimes.

A candidate is rejected when it is better explained by:

- sensor/encoder-like jumps;
- the configured sensor-shadow window;
- local background/common-mode motion;
- insufficient pair agreement;
- same-site synchronous events;
- weak event confidence.

Only surviving high-confidence events are allowed to move the prediction away from the constant prior.

In [3]:
t0 = time.perf_counter()

# Build every published turbine so the site-common-mode veto uses the full
# unlabeled site context rather than only the labelled turbines.
all_bundles = {
    turbine: build_turbine_bundle(
        turbine,
        binned,
        layout,
        FLEET,
        RELATIVE,
        MODEL,
    )
    for turbine in turbines
}

all_bundles = apply_site_common_mode_veto(
    all_bundles,
    MODEL,
)

bundles = {
    turbine: all_bundles[turbine]
    for turbine in TRAIN + TARGETS
}

MODEL_BUILD_SECONDS = time.perf_counter() - t0

# Pair reliability summary.
pair_rows = []
for turbine, bundle in bundles.items():
    table = bundle["pair_diagnostics"].reset_index(names="neighbour")
    table.insert(0, "turbine", turbine)
    pair_rows.append(
        table[
            [
                "turbine",
                "neighbour",
                "coverage",
                "pair_mad",
                "reliability_weight",
                "usable",
            ]
        ]
    )

pair_diagnostics = pd.concat(pair_rows, ignore_index=True)
display(pair_diagnostics.round(3))

# Keep the full event table in memory, but show only accepted events and
# important vetoes in the notebook.
event_rows = []
for turbine, bundle in bundles.items():
    table = bundle["boundaries"]
    if table is None or table.empty:
        continue

    out = table.reset_index(names="date")
    out.insert(0, "turbine", turbine)
    event_rows.append(out)

event_diagnostics = (
    pd.concat(event_rows, ignore_index=True)
    if event_rows
    else pd.DataFrame()
)

if not event_diagnostics.empty:
    important_reasons = {
        "yaw",
        "pelt_yaw",
        "sensor",
        "sensor_shadow",
        "site_common_mode",
        "low_confidence_prior",
    }

    event_view = event_diagnostics[
        event_diagnostics["accepted"]
        | event_diagnostics["reason"].isin(important_reasons)
    ][
        [
            "turbine",
            "date",
            "source",
            "change",
            "z",
            "pair_agreement",
            "event_confidence",
            "shrinkage",
            "reason",
            "accepted",
        ]
    ].copy()

    numeric = event_view.select_dtypes(include=[np.number]).columns
    event_view[numeric] = event_view[numeric].round(3)
    display(event_view.sort_values(["turbine", "date"]))

print(f"Relative-heading model build: {MODEL_BUILD_SECONDS:.1f}s")

,turbine,neighbour,coverage,pair_mad,reliability_weight,usable
0,PPP_WTG12,PPP_WTG11,0.625,3.555,0.256,True
1,PPP_WTG12,PPP_WTG13,0.653,6.055,0.129,True
2,PPP_WTG12,PPP_WTG14,0.614,8.402,0.043,True
3,PPP_WTG13,PPP_WTG14,0.817,5.033,0.275,True
4,PPP_WTG13,PPP_WTG12,0.707,6.192,0.124,True
5,PPP_WTG13,PPP_WTG11,0.832,7.804,0.058,True
6,PPP_WTG13,PPP_WTG08,0.843,7.669,0.039,True
7,PPP_WTG14,PPP_WTG13,0.774,5.210,0.253,True
8,PPP_WTG14,PPP_WTG12,0.608,8.662,0.035,True
9,PPP_WTG14,PPP_WTG08,0.768,6.125,0.059,True


,turbine,date,source,change,z,pair_agreement,event_confidence,shrinkage,reason,accepted
6,PPP_WTG12,2023-07-09,pelt,-33.198,23.921,0.000,0.000,0.000,sensor,False
7,PPP_WTG12,2023-07-17,rolling,-32.451,34.241,0.750,0.263,0.000,sensor,False
8,PPP_WTG12,2023-08-07,rolling,-8.654,8.388,0.900,0.000,0.000,sensor_shadow,False
9,PPP_WTG12,2023-08-08,pelt,-8.396,4.798,0.900,0.000,0.000,sensor_shadow,False
13,PPP_WTG12,2024-03-11,rolling,-6.030,3.093,0.699,0.000,0.000,site_common_mode,False
16,PPP_WTG13,2023-02-19,rolling,-3.614,7.227,0.751,0.321,0.321,yaw,True
20,PPP_WTG13,2023-07-27,pelt,-3.603,2.236,0.884,0.436,0.436,pelt_yaw,True
23,PPP_WTG13,2024-03-10,rolling,-6.261,5.067,0.671,0.000,0.000,site_common_mode,False
31,PPP_WTG14,2023-06-05,rolling,-3.435,3.259,0.680,0.142,0.000,low_confidence_prior,False
34,PPP_WTG14,2024-03-11,rolling,6.584,3.226,0.839,0.000,0.000,site_common_mode,False


Relative-heading model build: 25.2s


## 4. Strict turbine-level LOTO validation

Each fold:

1. holds out one labelled turbine;
2. estimates \(C\) and any eligible global \($\beta$\) using only the other labelled turbines;
3. uses the holdout turbine only through its unlabeled SCADA-derived prior/state features;
4. compares the state-aware prediction with the same-fold constant prior.

Because only a few reliable labelled state transitions survive, the model deliberately falls back to the physical prior \($\beta=1$\) instead of fitting an unstable slope.

The notebook reports MAE/RMSE. Legacy micro-step boundary metrics returned by the helper are intentionally not used for model selection.

In [4]:
t0 = time.perf_counter()
rows = []

for holdout in TRAIN:
    train_ids = [
        turbine
        for turbine in TRAIN
        if turbine != holdout
    ]

    C, beta = fit_global_beta(
        train_ids,
        bundles,
        labels,
        theta_star,
        MODEL,
    )

    prediction = predict_from_bundle(
        holdout,
        bundles[holdout],
        C,
        beta,
        theta_star,
    )

    metrics = score_prediction(
        prediction,
        labels[holdout],
        C - theta_star[holdout],
    )

    boundary_table = bundles[holdout]["boundaries"]
    accepted = (
        boundary_table.loc[
            boundary_table["accepted"].fillna(False)
        ]
        if len(boundary_table)
        else pd.DataFrame()
    )

    rows.append(
        {
            "holdout": holdout,
            "mae": metrics["mae"],
            "rmse": metrics["rmse"],
            "constant_mae": metrics["constant_mae"],
            "constant_rmse": metrics["constant_rmse"],
            "beta": beta,
            "n_states": int(
                bundles[holdout]["observables"]["cluster"].nunique()
            ),
            "accepted_boundaries": ", ".join(
                pd.Timestamp(date).date().isoformat()
                for date in accepted.index
            )
            or "none",
        }
    )

LOTO_SECONDS = time.perf_counter() - t0
loto = pd.DataFrame(rows)

display(loto.round(3))

macro = pd.DataFrame(
    {
        "state_aware": [
            loto["mae"].mean(),
            loto["rmse"].mean(),
        ],
        "constant_prior": [
            loto["constant_mae"].mean(),
            loto["constant_rmse"].mean(),
        ],
    },
    index=["MAE", "RMSE"],
)

display(macro.round(3))

mae_gain = 1.0 - macro.loc["MAE", "state_aware"] / macro.loc["MAE", "constant_prior"]
rmse_gain = 1.0 - macro.loc["RMSE", "state_aware"] / macro.loc["RMSE", "constant_prior"]

print(
    f"Macro improvement: MAE {mae_gain:.1%}, RMSE {rmse_gain:.1%}"
)
print(f"LOTO scoring: {LOTO_SECONDS:.2f}s")

,holdout,mae,rmse,constant_mae,constant_rmse,beta,n_states,accepted_boundaries
0,PPP_WTG12,0.357,0.608,0.357,0.608,1.0,1,none
1,PPP_WTG13,0.608,0.979,1.004,1.387,1.0,3,"2023-02-19, 2023-07-27"
2,PPP_WTG14,0.206,0.262,0.206,0.262,1.0,1,none


,state_aware,constant_prior
MAE,0.390,0.522
RMSE,0.616,0.752


Macro improvement: MAE 25.3%, RMSE 18.1%
LOTO scoring: 0.04s


## 5. Final fit and unlabeled target diagnostics

The final calibration uses all three labelled training turbines.

Target diagnostics below are **not validation scores**. They are shown to make deployment behaviour auditable:

- accepted state dates;
- detector source and confidence;
- correction range around the constant prior;
- resulting prediction range.

This is especially important because the targets may contain larger unlabeled state structure than the training turbines.

In [5]:
FINAL_C, FINAL_BETA = fit_global_beta(
    TRAIN,
    bundles,
    labels,
    theta_star,
    MODEL,
)

predictions = {
    turbine: predict_from_bundle(
        turbine,
        bundles[turbine],
        FINAL_C,
        FINAL_BETA,
        theta_star,
    )
    for turbine in TRAIN + TARGETS
}

diagnostic_rows = []

for turbine in TRAIN + TARGETS:
    bundle = bundles[turbine]
    boundary_table = bundle["boundaries"]

    accepted = (
        boundary_table.loc[
            boundary_table["accepted"].fillna(False)
        ]
        if len(boundary_table)
        else pd.DataFrame()
    )

    correction = bundle["observables"][
        "relative_prior_correction"
    ].fillna(0.0)

    pred = predictions[turbine]["prediction"]

    diagnostic_rows.append(
        {
            "turbine": turbine,
            "constant_prior": FINAL_C - theta_star[turbine],
            "accepted_boundaries": ", ".join(
                pd.Timestamp(date).date().isoformat()
                for date in accepted.index
            )
            or "none",
            "accepted_sources": ", ".join(
                accepted["source"].astype(str).tolist()
            )
            if len(accepted)
            else "none",
            "mean_event_confidence": (
                float(accepted["event_confidence"].mean())
                if len(accepted)
                else 0.0
            ),
            "correction_range": float(
                correction.max() - correction.min()
            ),
            "prediction_min": float(pred.min()),
            "prediction_max": float(pred.max()),
        }
    )

deployment_diagnostics = pd.DataFrame(diagnostic_rows)

display(deployment_diagnostics.round(3))

print(f"Final C = {FINAL_C:.3f}")
print(f"Final beta = {FINAL_BETA:.3f}")
print(
    {
        "load_and_B0_seconds": round(LOAD_SECONDS, 2),
        "relative_model_seconds": round(MODEL_BUILD_SECONDS, 2),
        "LOTO_seconds": round(LOTO_SECONDS, 2),
    }
)

# Basic release sanity checks.
for turbine, pred in predictions.items():
    values = pred["prediction"].to_numpy(dtype=float)
    assert np.isfinite(values).all(), f"Non-finite prediction for {turbine}"
    assert np.max(np.abs(values)) <= 90.0, f"Prediction out of bounds for {turbine}"

print("Release sanity checks passed.")

,turbine,constant_prior,accepted_boundaries,accepted_sources,mean_event_confidence,correction_range,prediction_min,prediction_max
0,PPP_WTG12,-2.642,none,none,0.000,0.000,-2.642,-2.642
1,PPP_WTG13,-6.437,"2023-02-19, 2023-07-27","rolling, pelt",0.379,1.631,-7.764,-6.133
2,PPP_WTG14,-1.731,none,none,0.000,0.000,-1.731,-1.731
3,PPP_WTG17,-3.602,"2023-06-25, 2023-10-15, 2024-01-07","rolling, rolling, rolling",0.512,5.258,-6.680,-1.422
4,SSS_WTG06,-4.572,"2023-05-28, 2023-09-24, 2024-10-13","rolling, rolling, rolling",0.542,4.434,-8.162,-3.729


Final C = -6.085
Final beta = 1.000
{'load_and_B0_seconds': 162.21, 'relative_model_seconds': 25.21, 'LOTO_seconds': 0.04}
Release sanity checks passed.


## 6. Interpretation and limitations

The current model is intentionally conservative.

**What the validation supports**

- The long-term B0 prior is already strong for turbines whose yaw level is effectively static.
- Persistent relative-heading states add value when there is real temporal structure, as seen on WTG13.
- Sensor/reference events must be excluded across all detector sources; otherwise a state detector can turn an encoder event into a false yaw correction.
- The available labels do not support learning a complex amplitude mapping, so \($\beta=1$\) remains the preferred physical prior when transition evidence is sparse.

**What is not established**

- Good LOTO performance on three PPP turbines does not prove the magnitude of unlabeled PPP17/SSS06 state corrections.
- SSS06 is a cross-site transfer and remains the highest domain-shift risk.
- The model should therefore be treated as a constant-prior model with sparse corrections, not as a general daily yaw estimator.

## 7. Validation results visualization

In [6]:
# ============================================================
# Blind validation / final-target state summaries
# ============================================================

def prediction_state_summary(turbine, predictions):
    pred = predictions[turbine].copy()

    pred["date"] = pd.to_datetime(pred.index)
    pred["cluster"] = pred["cluster"].astype(int)

    summary = (
        pred.groupby("cluster")
        .agg(
            start=("date", "min"),
            end=("date", "max"),
            days=("date", "size"),
            yaw_prediction=("prediction", "median"),
            constant_prior=("constant_prior", "median"),
            relative_correction=(
                "relative_prior_correction",
                "median",
            ),
        )
        .reset_index()
    )

    return summary


print("PPP_WTG17 — blind validation prediction")
display(
    prediction_state_summary(
        "PPP_WTG17",
        predictions,
    ).round(3)
)

print("SSS_WTG06 — blind final-target prediction")
display(
    prediction_state_summary(
        "SSS_WTG06",
        predictions,
    ).round(3)
)

PPP_WTG17 — blind validation prediction


C:\Users\HJL\AppData\Local\Temp\ipykernel_16824\418222005.py:35: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ).round(3)


,cluster,start,end,days,yaw_prediction,constant_prior,relative_correction
0,0,2023-01-01,2023-06-24,175,-5.246,-3.602,1.644
1,1,2023-06-25,2023-10-14,112,-5.732,-3.602,2.130
2,2,2023-10-15,2024-01-06,84,-6.680,-3.602,3.078
3,3,2024-01-07,2024-12-31,360,-1.422,-3.602,-2.180


SSS_WTG06 — blind final-target prediction


C:\Users\HJL\AppData\Local\Temp\ipykernel_16824\418222005.py:43: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ).round(3)


,cluster,start,end,days,yaw_prediction,constant_prior,relative_correction
0,0,2023-01-01,2023-05-27,147,-4.079,-4.572,-0.494
1,1,2023-05-28,2023-09-23,119,-8.162,-4.572,3.590
2,2,2023-09-24,2024-10-12,385,-3.826,-4.572,-0.746
3,3,2024-10-13,2024-12-31,80,-3.729,-4.572,-0.843


In [7]:
validation_preview = (
    predictions["PPP_WTG17"]
    [["cluster", "prediction"]]
    .rename(
        columns={
            "prediction": "yaw_misalignment_deg",
        }
    )
    .copy()
)

validation_preview.insert(
    0,
    "date",
    validation_preview.index.strftime("%Y-%m-%d"),
)

validation_preview.insert(
    0,
    "turbine_id",
    "PPP_WTG17",
)

display(validation_preview.head())
display(validation_preview.tail())

assert len(validation_preview) == 731
assert validation_preview[
    "yaw_misalignment_deg"
].notna().all()

,turbine_id,date,cluster,yaw_misalignment_deg
2023-01-01,PPP_WTG17,2023-01-01,0,-5.245909
2023-01-02,PPP_WTG17,2023-01-02,0,-5.245909
2023-01-03,PPP_WTG17,2023-01-03,0,-5.245909
2023-01-04,PPP_WTG17,2023-01-04,0,-5.245909
2023-01-05,PPP_WTG17,2023-01-05,0,-5.245909


,turbine_id,date,cluster,yaw_misalignment_deg
2024-12-27,PPP_WTG17,2024-12-27,3,-1.422174
2024-12-28,PPP_WTG17,2024-12-28,3,-1.422174
2024-12-29,PPP_WTG17,2024-12-29,3,-1.422174
2024-12-30,PPP_WTG17,2024-12-30,3,-1.422174
2024-12-31,PPP_WTG17,2024-12-31,3,-1.422174
